In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import load_model
import datetime
import re


TIMESERIES_DIR = 'measured_data\progress'
TARGETS_FILE = 'training_progress_data_ready.csv'
MAX_TIMESTEPS = 2500

FEATURE_COLS = ['A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7']
TARGET_COLS = ['x', 'y', 'z', 'Gewicht']

def load_data(timeseries_dir, targets_file):

    targets_df = pd.read_csv(targets_file)

    file_ids = targets_df['source_file'].values 
    
    X_list = []
    y_list = []
    
    for fid in file_ids:
        filepath = os.path.join(timeseries_dir, f"{fid}")
        if not os.path.exists(filepath):
            print(f"Warnung: Datei {filepath} nicht gefunden.")
            continue

        df_ts = pd.read_csv(filepath)
        sequence = df_ts[FEATURE_COLS].values
        X_list.append(sequence)

        targets = targets_df[targets_df['source_file'] == fid][TARGET_COLS].values[0]
        y_list.append(targets)

    X_padded = pad_sequences(X_list, maxlen=MAX_TIMESTEPS, dtype='float32', padding='post', truncating='post')
    y_array = np.array(y_list, dtype='float32')
    
    return X_padded, y_array

X, y = load_data(TIMESERIES_DIR, TARGETS_FILE)
print(f"Shape von X (Samples, Timesteps, Features): {X.shape}")
print(f"Shape von y (Samples, Targets): {y.shape}")

Shape von X (Samples, Timesteps, Features): (170, 2500, 7)
Shape von y (Samples, Targets): (170, 4)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)  # 20% von 80% = 16%

scaler_X = RobustScaler()

num_samples_train, timesteps, num_features = X_train.shape
X_train_reshaped = X_train.reshape(-1, num_features)
X_train_scaled = scaler_X.fit_transform(X_train_reshaped)
X_train_scaled = X_train_scaled.reshape(num_samples_train, timesteps, num_features)

num_samples_val = X_val.shape[0]
X_val_reshaped = X_val.reshape(-1, num_features)
X_val_scaled = scaler_X.transform(X_val_reshaped)
X_val_scaled = X_val_scaled.reshape(num_samples_val, timesteps, num_features)

num_samples_test = X_test.shape[0]
X_test_reshaped = X_test.reshape(-1, num_features)
X_test_scaled = scaler_X.transform(X_test_reshaped)
X_test_scaled = X_test_scaled.reshape(num_samples_test, timesteps, num_features)

print("Daten wurden erfolgreich gesplittet und skaliert.")

Daten wurden erfolgreich gesplittet und skaliert.


In [ ]:
def build_gru_model(timesteps, features, output_dim=4):
    model = Sequential([
        GRU(128, return_sequences=True, input_shape=(timesteps, features)),
        Dropout(0.2),
        BatchNormalization(),

        GRU(64, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),

        Dense(32, activation='relu'),
        Dense(output_dim) # 4 Neuronen für x, y, z, Gewicht
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_gru_model(MAX_TIMESTEPS, 7)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 2500, 128)         52608     
                                                                 
 dropout (Dropout)           (None, 2500, 128)         0         
                                                                 
 batch_normalization (BatchN  (None, 2500, 128)        512       
 ormalization)                                                   
                                                                 
 gru_1 (GRU)                 (None, 64)                37248     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 batch_normalization_1 (Batc  (None, 64)               256       
 hNormalization)                                        

In [ ]:
MODEL_PATH = 'best_gru_model.h5'

if os.path.exists(MODEL_PATH):
    print(f"Lade bestehende Modellgewichte aus {MODEL_PATH} um weiterzutrainieren...")
    model.load_weights(MODEL_PATH)

checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH, 
    monitor='val_loss', 
    save_best_only=True, 
    save_weights_only=False,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=200,
    restore_best_weights=True
)

print("Starte Training...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=1000,
    batch_size=32,
    validation_data=(X_val_scaled, y_val)
    callbacks=[checkpoint, early_stopping]
)

Lade bestehende Modellgewichte aus best_gru_model.h5 um weiterzutrainieren...
Starte Training...
Epoch 1/1000
4/4 [==============================] - ETA: 0s - loss: 193.2192 - mae: 10.3700
Epoch 1: val_loss improved from inf to 87.50921, saving model to best_gru_model.h5
4/4 [==============================] - 1s 218ms/step - loss: 193.2192 - mae: 10.3700 - val_loss: 87.5092 - val_mae: 6.5939
Epoch 2/1000
4/4 [==============================] - ETA: 0s - loss: 171.5398 - mae: 9.6357
Epoch 2: val_loss did not improve from 87.50921
4/4 [==============================] - 1s 195ms/step - loss: 171.5398 - mae: 9.6357 - val_loss: 144.3560 - val_mae: 8.7586
Epoch 3/1000
4/4 [==============================] - ETA: 0s - loss: 212.8494 - mae: 10.9040
Epoch 3: val_loss did not improve from 87.50921
4/4 [==============================] - 1s 192ms/step - loss: 212.8494 - mae: 10.9040 - val_loss: 117.8699 - val_mae: 8.1904
Epoch 4/1000
4/4 [==============================] - ETA: 0s - loss: 179.6274 - 

In [ ]:
MODEL_PATH = 'gru_robust_arch_64-32-16_lr_0.001_l2_0.0_valLoss_72.2675.keras'


best_model = load_model(MODEL_PATH)

y_pred = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"\nTest MAE (Mean Absolute Error): {mae:.4f}")
print(f"Test MSE (Mean Squared Error): {mse:.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test[:, 3], y_pred[:, 3], alpha=0.6, color='blue')
plt.plot([y_test[:, 3].min(), y_test[:, 3].max()], [y_test[:, 3].min(), y_test[:, 3].max()], 'r--')
plt.xlabel('Tatsächliches Gewicht')
plt.ylabel('Vorhergesagtes Gewicht')
plt.title('Vorhersage vs. Wahrheit (Gewicht)')
plt.grid(True)
plt.show()

TypeError: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 134367038852608}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 2500, 7], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer', 'optional': False}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'GRU', 'config': {'name': 'gru', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 13436

In [ ]:
model = load_model(MODEL_PATH)
y_pred = model.predict(X_test_scaled)

results = pd.DataFrame(y_test, columns=["x", "y", "z", "Gewicht"])

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred","Gewicht_pred"]
)

results[["x_pred", "y_pred", "z_pred","Gewicht_pred"]] = y_pred_df

results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

# df existiert in diesem Kontext nicht, da die Daten in der load_data Funktion geladen wurden.
# Daher wird dieser Schritt hier übersprungen oder muss über die load_data Funktion zurückgegeben werden.
# results = pd.concat([results, df.loc[y_test.index, ["source_file"]]], axis=1)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = model.__class__.__name__

try:
    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)
    opt_name = model.optimizer.__class__.__name__
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    param_str = "keras_nn"

filename = f"Auswertung_test_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)
#results.to_csv(filename, index=True, sep=";", decimal=",")

print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

2/2 [==============================] - 1s 71ms/step
Auswertung Testdatensatz:
  - MAE x: 11.3151
  - MAE y: 12.2870
  - MAE z: 5.5633
  - MAE Gewicht: 5.6250
  - MSE x: 219.1024
  - MSE y: 235.5760
  - MSE z: 60.3674
  - MSE Gewicht: 46.5009

[INFO] Datei erfolgreich gespeichert unter: Auswertung_test_Sequential_Arch_128-64-32-4_Adam_lr_0.0010000000474974513_20260622_120011.csv


In [ ]:
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht
0,"81,39","46,35","20,00","58,10","71,45","56,56","16,63","53,79","9,94","10,21","3,37","4,31","98,784386","104,233101","11,354489","18,552509"
1,"53,33","91,00","30,00","443,80","81,74","69,40","30,78","456,04","28,41","21,60","0,78","12,24","807,019043","466,551025","0,605812","149,806900"
2,"54,50","47,50","20,00","43,90","71,33","41,72","17,58","44,40","16,83","5,78","2,42","0,50","283,086426","33,355091","5,837399","0,251490"
3,"53,33","91,00","30,00","443,80","82,05","69,85","30,97","457,54","28,72","21,15","0,97","13,74","824,841553","447,179596","0,949588","188,686707"
4,"46,35","81,39","20,00","58,10","70,32","47,88","21,38","50,91","23,97","33,51","1,38","7,19","574,451965","1122,757935","1,906367","51,721207"
5,"120,00","12,50","17,50","71,60","121,85","18,60","26,80","75,84","1,85","6,10","9,30","4,24","3,410592","37,174294","86,457245","17,978682"
6,"12,50","120,00","17,50","71,60","18,04","99,48","17,96","62,86","5,54","20,52","0,46","8,74","30,694950","421,168579","0,209043","76,457863"
7,"31,75","25,00","50,00","127,80","35,09","38,09","37,35","123,84","3,34","13,09","12,65","3,96","11,179133","171,478165","159,917068","15,692834"
8,"47,50","32,50","17,25","19,30","43,28","36,22","30,84","25,51","4,22","3,72","13,59","6,21","17,840652","13,812651","184,683899","38,596077"
9,"87,50","47,50","20,00","51,70","68,88","50,18","32,53","51,80","18,62","2,68","12,53","0,10","346,657349","7,163725","156,940552","0,009793"


In [ ]:
y_pred = model.predict(X_train_scaled)

results = pd.DataFrame(y_train, columns=["x", "y", "z", "Gewicht"])

y_pred_df = pd.DataFrame(
    y_pred, columns=["x_pred", "y_pred", "z_pred", "Gewicht_pred"]
)

results[["x_pred", "y_pred", "z_pred", "Gewicht_pred"]] = y_pred_df

results["abs_error_x"] = np.abs(results["x_pred"] - results["x"])
results["abs_error_y"] = np.abs(results["y_pred"] - results["y"])
results["abs_error_z"] = np.abs(results["z_pred"] - results["z"])
results["abs_error_Gewicht"] = np.abs(results["Gewicht_pred"] - results["Gewicht"])

results["mse_x"] = (results["x_pred"] - results["x"]) ** 2
results["mse_y"] = (results["y_pred"] - results["y"]) ** 2
results["mse_z"] = (results["z_pred"] - results["z"]) ** 2
results["mse_Gewicht"] = (results["Gewicht_pred"] - results["Gewicht"]) ** 2

# df ist hier nicht definiert, also überspringen wir das.
# results = pd.concat([results, df.loc[y_train.index, ["source_file"]]], axis=1)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

model_name = model.__class__.__name__


try:

    dense_units = [
        str(layer.units) for layer in model.layers if hasattr(layer, "units")
    ]
    arch_str = "-".join(dense_units)
    opt_name = model.optimizer.__class__.__name__
    try:
        lr = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
    except Exception:
        try:
            lr = float(model.optimizer.learning_rate.numpy())
        except Exception:
            lr = "unknown"
    param_str = f"Arch_{arch_str}_{opt_name}_lr_{lr}"

except Exception:
    param_str = "keras_nn"

filename = f"Auswertung_train_{model_name}_{param_str}_{timestamp}.csv"
filename = re.sub(r'[\\/*?:"<>| ]', "_", filename)
#results.to_csv(filename, index=True, sep=";", decimal=",")

print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

print(f"\n[INFO] Datei erfolgreich gespeichert unter: {filename}")

4/4 [==============================] - 0s 76ms/step
Auswertung Trainingsdatensatz:
  - MAE x: 9.7102
  - MAE y: 10.3482
  - MAE z: 6.3780
  - MAE Gewicht: 4.3716
  - MSE x: 154.1405
  - MSE y: 176.1411
  - MSE z: 85.2442
  - MSE Gewicht: 32.6083

[INFO] Datei erfolgreich gespeichert unter: Auswertung_train_Sequential_Arch_128-64-32-4_Adam_lr_0.0010000000474974513_20260622_120040.csv


In [ ]:
results.style.format(
    {
        "x": "{:.2f}",
        "x_pred": "{:.2f}",
        "abs_error_x": "{:.2f}",
        "y": "{:.2f}",
        "y_pred": "{:.2f}",
        "abs_error_y": "{:.2f}",
        "z": "{:.2f}",
        "z_pred": "{:.2f}",
        "abs_error_z": "{:.2f}",
        "Gewicht": "{:.2f}",
        "Gewicht_pred": "{:.2f}",
        "abs_error_Gewicht": "{:.2f}",
    },
    decimal=",",
).background_gradient(
    subset=["abs_error_x", "abs_error_y", "abs_error_z","abs_error_Gewicht"], cmap="Reds"
)

,x,y,z,Gewicht,x_pred,y_pred,z_pred,Gewicht_pred,abs_error_x,abs_error_y,abs_error_z,abs_error_Gewicht,mse_x,mse_y,mse_z,mse_Gewicht
0,"87,50","47,50","20,00","51,70","70,44","54,64","20,16","50,62","17,06","7,14","0,16","1,08","291,045074","50,937492","0,025307","1,163670"
1,"87,50","47,50","20,00","51,70","78,96","54,42","22,10","57,08","8,54","6,92","2,10","5,38","72,875465","47,819084","4,407687","28,922005"
2,"47,50","32,50","17,25","19,30","40,22","28,26","13,64","21,75","7,28","4,24","3,61","2,45","52,987831","18,018606","13,056408","5,999980"
3,"50,00","25,00","31,75","127,80","36,91","37,47","35,39","124,37","13,09","12,47","3,64","3,43","171,269913","155,451279","13,231217","11,793915"
4,"64,61","46,35","20,00","58,10","61,09","60,18","18,16","51,12","3,52","13,83","1,84","6,98","12,418429","191,217880","3,380493","48,747555"
5,"91,00","53,33","30,00","443,80","82,01","69,79","30,92","457,30","8,99","16,46","0,92","13,50","80,888115","270,914246","0,839336","182,133011"
6,"32,50","47,50","17,25","19,30","29,59","44,35","10,75","17,83","2,91","3,15","6,50","1,47","8,467400","9,939893","42,254936","2,154740"
7,"47,50","20,00","87,50","51,70","32,95","60,07","62,88","43,54","14,55","40,07","24,62","8,16","211,579391","1605,971436","605,991455","66,614235"
8,"31,75","50,00","25,00","127,80","33,61","35,90","35,09","116,61","1,86","14,10","10,09","11,19","3,447267","198,943054","101,835739","125,324257"
9,"20,00","87,50","47,50","51,70","40,23","73,70","49,96","53,66","20,23","13,80","2,46","1,96","409,079407","190,533585","6,063576","3,855306"
